# Retail Sales Performance Analysis
Ensure your dataset is placed at `../data/raw/retail_sales.csv` (or generated via `../scripts/generate_data.py`) before running the cells below.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
RAW_PATH = BASE_DIR / "data" / "raw" / "retail_sales.csv"
PROCESSED_PATH = BASE_DIR / "data" / "processed" / "retail_sales_clean.csv"
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

# 1. Load the dataset
df = pd.read_csv(RAW_PATH)

print("Shape:", df.shape)
display(df.head())

In [ ]:
# 2. Data cleaning & feature engineering
df = df.drop_duplicates()

# Parse dates safely
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df = df.dropna(subset=["Order_Date", "Sales"])

# Fill missing quantity with 0 if needed
if "Quantity" in df.columns:
    df["Quantity"] = df["Quantity"].fillna(0)

# Profit margin (%) if Profit column exists
if "Profit" in df.columns:
    df["Profit_Margin_%"] = (df["Profit"] / df["Sales"].replace(0, pd.NA)) * 100

df["Year_Month"] = df["Order_Date"].dt.to_period("M").astype(str)

# Save cleaned dataset
df.to_csv(PROCESSED_PATH, index=False)
print(f"Data cleaned and saved to {PROCESSED_PATH}")

In [ ]:
# 3. Core KPIs
total_sales = df["Sales"].sum()
total_orders = len(df)
avg_order_value = df["Sales"].mean()

print(f"Total Sales: {total_sales:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Average Order Value (AOV): {avg_order_value:,.2f}")

if "Profit" in df.columns:
    total_profit = df["Profit"].sum()
    overall_margin = (total_profit / total_sales) * 100 if total_sales else 0
    print(f"Total Profit: {total_profit:,.2f} ({overall_margin:.2f}% margin)")

In [ ]:
# 4. Monthly sales trend
monthly_sales = df.groupby("Year_Month", as_index=False)["Sales"].sum()

plt.figure(figsize=(10, 4))
plt.plot(monthly_sales["Year_Month"], monthly_sales["Sales"], marker="o")
plt.title("Monthly Retail Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "01_monthly_sales_trend.png", dpi=150)
plt.show()

In [ ]:
# 5. Top categories & regions
if "Category" in df.columns:
    cat_perf = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
    print("\nSales by Category:\n", cat_perf)

    ax = cat_perf.plot(
        kind="bar", figsize=(8, 4), title="Sales by Category", ylabel="Total Sales"
    )
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "02_sales_by_category.png", dpi=150)
    plt.show()

if "Region" in df.columns:
    reg_perf = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
    print("\nSales by Region:\n", reg_perf)